# TridiArduSim async AI gateway starter

This notebook receives selected simulator camera frames and returns short joint-target action chunks. It uses asynchronous 5–10 Hz requests; motion continues in the browser between replies. The included policy holds the current joint pose as a connectivity check. Replace `infer_action` with your own model.

Use only a trusted tunnel URL. The tunnel credential is entered with a hidden prompt and is never sent to the simulator. Do not put credentials in notebook output.

In [ ]:
%pip -q install websockets pyngrok opencv-python-headless
import asyncio, base64, getpass, itertools, json
import cv2, numpy as np, websockets
from pyngrok import ngrok

ngrok.set_auth_token(getpass.getpass('ngrok auth token (hidden): '))
print('Packages ready. Start the gateway in the next cell.')

In [ ]:
action_sequence = itertools.count()

async def infer_action(image_bgr, robot):
    # Replace this hold policy with your own model. Input is a 320x240
    # BGR OpenCV image; outputs are joint targets in radians.
    return [float(value) for value in robot.get('jointsRad', [])]

async def handle_client(websocket):
    async for raw in websocket:
        if len(raw) > 400_000:
            continue
        try:
            message = json.loads(raw)
            if message.get('type') == 'hello':
                await websocket.send(json.dumps({'type': 'ready', 'protocolVersion': 1}))
                continue
            if message.get('type') != 'camera_frame' or message.get('protocolVersion') != 1:
                continue
            encoded = base64.b64decode(message['imageJpegBase64'], validate=True)
            image = cv2.imdecode(np.frombuffer(encoded, dtype=np.uint8), cv2.IMREAD_COLOR)
            if image is None or image.shape[1] > 320 or image.shape[0] > 320:
                continue
            robot = message.get('robot', {})
            targets = await infer_action(image, robot)
            chunk = {
                'type': 'action_chunk',
                'sequence': next(action_sequence),
                'validForMs': 200,
                'joints': [
                    {'robotId': robot['id'], 'jointIndex': index, 'targetRad': float(value)}
                    for index, value in enumerate(targets)
                ],
            }
            if chunk['joints']:
                await websocket.send(json.dumps(chunk, separators=(',', ':')))
        except Exception as error:
            print('Skipped invalid frame:', type(error).__name__)

server = await websockets.serve(handle_client, '0.0.0.0', 8765, max_size=400_000)
tunnel = ngrok.connect(8765, 'http')
gateway_url = tunnel.public_url.replace('https://', 'wss://').replace('http://', 'ws://')
print('Paste this URL into TridiArduSim → Async AI Gateway:')
print(gateway_url)
await asyncio.Future()  # Keep the Colab runtime and tunnel alive.